# 01 · Data Overview & Data Quality

**Project:** Predicting whether a powerlifting attempt will be made (OpenPowerlifting, `openpowerlifting.csv`)

This notebook loads the raw data, documents every column, and checks data quality.
It does **not** change the data; all cleaning decisions are recorded here and applied later by the fixed rules in `src/clean.py`.

In [1]:
import sys
sys.path.append('../src')
import numpy as np
import pandas as pd
from clean import raw_path, load_raw, USECOLS, ATTEMPT_COLS, BEST_COLS

pd.set_option('display.max_columns', None)
RAW = raw_path()
print('Reading', RAW)

Reading ../data/openpowerlifting.csv


## 1. Load the data

The file is 828 MB, so we first look at a few rows of **all 42 columns**, and later load only the columns the project needs.

In [2]:
preview = pd.read_csv(RAW, nrows=5)
print(preview.shape[1], 'columns')
preview

42 columns


,Name,Sex,Event,Equipment,Age,AgeClass,BirthYearClass,Division,BodyweightKg,WeightClassKg,Squat1Kg,Squat2Kg,Squat3Kg,Squat4Kg,Best3SquatKg,Bench1Kg,Bench2Kg,Bench3Kg,Bench4Kg,Best3BenchKg,Deadlift1Kg,Deadlift2Kg,Deadlift3Kg,Deadlift4Kg,Best3DeadliftKg,TotalKg,Place,Dots,Wilks,Glossbrenner,Goodlift,Tested,Country,State,Federation,ParentFederation,Date,MeetCountry,MeetState,MeetTown,MeetName,Sanctioned
0,E.S. Denisenko,F,B,Raw,28.5,24-34,24-39,Open,67.3,NaN,NaN,NaN,NaN,NaN,NaN,-40.0,-45.0,-45.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DQ,NaN,NaN,NaN,NaN,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
1,I.S. Lebetskaya,F,B,Raw,43.5,40-44,40-49,Open,73.2,NaN,NaN,NaN,NaN,NaN,NaN,80.0,85.0,90.0,NaN,90.0,NaN,NaN,NaN,NaN,NaN,90.0,1,88.80,86.89,76.50,70.06,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
2,K. Yakimovich,F,B,Raw,26.5,24-34,24-39,Open,60.6,NaN,NaN,NaN,NaN,NaN,NaN,40.0,42.5,45.0,NaN,45.0,NaN,NaN,NaN,NaN,NaN,45.0,2,49.57,49.79,43.98,38.42,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
3,A.G. Golneva,F,B,Raw,19.5,20-23,19-23,Juniors 17-21,50.3,NaN,NaN,NaN,NaN,NaN,NaN,32.5,35.0,-37.5,NaN,35.0,NaN,NaN,NaN,NaN,NaN,35.0,2,43.67,44.76,39.73,34.55,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes
4,E.V. Marunevskaya,F,B,Raw,19.5,20-23,19-23,Juniors 17-21,63.7,NaN,NaN,NaN,NaN,NaN,NaN,40.0,42.5,-45.0,NaN,42.5,NaN,NaN,NaN,NaN,NaN,42.5,1,45.40,45.26,39.96,35.25,NaN,Belarus,NaN,GSF-Belarus,NaN,2019-06-22,Belarus,NaN,Luninets,Bison Power Cup,Yes


## 2. Data dictionary

Each row is **one lifter at one meet** (in one division). Source: the dataset's own README (`openpowerlifting-2026-10-17/README.txt`).

| Column | Type | Description |
| --- | --- | --- |
| Name | text | Lifter's name; lifters with the same name are told apart with `#1`, `#2` |
| Sex | categorical | `M`, `F` or `Mx` (gender-neutral category) |
| Event | categorical | Lifts contested: `SBD` (full power), `BD`, `SD`, `SB`, `S`, `B` (bench only), `D` |
| Equipment | categorical | `Raw` (sleeves), `Wraps`, `Single-ply`, `Multi-ply`, `Unlimited`, `Straps` |
| Age | numeric | Age on meet day; `x.5` means the lifter is either x or x+1 (only birth year known) |
| AgeClass | categorical | Age class, e.g. `24-34` |
| BirthYearClass | categorical | Birth-year class used by the IPF, e.g. `19-23` |
| Division | text | Free-form division name, e.g. `Open`, `Juniors` |
| BodyweightKg | numeric | Bodyweight at weigh-in (kg) |
| WeightClassKg | categorical | Weight class, e.g. `83` or `120+` |
| Squat1Kg … Squat3Kg | numeric | Squat attempts 1–3 (kg); **a negative number = missed attempt** |
| Squat4Kg | numeric | 4th attempt: a record try that does not count toward the total |
| Best3SquatKg | numeric | Best successful squat of attempts 1–3 (**a result**) |
| Bench1Kg … Bench4Kg, Best3BenchKg | numeric | Same for the bench press |
| Deadlift1Kg … Deadlift4Kg, Best3DeadliftKg | numeric | Same for the deadlift |
| TotalKg | numeric | Sum of the three best lifts; empty if any lift was failed (**a result**) |
| Place | categorical | Placing, or `G` guest, `DQ` disqualified, `DD` doping DQ, `NS` no-show (**a result**) |
| Dots, Wilks, Glossbrenner, Goodlift | numeric | Bodyweight-adjusted scores of the total (**results**) |
| Tested | categorical | `Yes` if the category was drug-tested |
| Country, State | categorical | Lifter's home country / state |
| Federation | categorical | Federation that hosted the meet, e.g. `USAPL` |
| ParentFederation | categorical | Top sanctioning body, e.g. `IPF` |
| Date | date | Start date of the meet |
| MeetCountry, MeetState, MeetTown | categorical | Where the meet was held |
| MeetName | text | Name of the meet (without year or federation) |
| Sanctioned | categorical | `No` for unsanctioned meets (they don't count for records) |

## 3. Data types and null values (all 42 columns)

To check every column without loading the whole file at once, we read it in chunks of 500,000 rows and add up the counts.

In [3]:
null_counts, rows = None, 0
for chunk in pd.read_csv(RAW, chunksize=500_000, low_memory=False):
    counts = chunk.isna().sum()
    null_counts = counts if null_counts is None else null_counts + counts
    rows += len(chunk)
types = chunk.dtypes

nulls = pd.DataFrame({'dtype': types.astype(str), 'nulls': null_counts, 'null_%': (null_counts / rows * 100).round(1)})
print(f'{rows:,} rows x {len(nulls)} columns')
nulls.sort_values('null_%', ascending=False)

4,050,248 rows x 42 columns


,dtype,nulls,null_%
Squat4Kg,float64,4040230,99.8
Bench4Kg,float64,4025492,99.4
Deadlift4Kg,float64,4021895,99.3
State,str,3184538,78.6
Squat3Kg,float64,2766366,68.3
Squat2Kg,float64,2734760,67.5
Squat1Kg,float64,2720903,67.2
Deadlift3Kg,float64,2639493,65.2
Deadlift2Kg,float64,2587780,63.9
Deadlift1Kg,float64,2562645,63.3


In [4]:
num_cols = [c for c in nulls.index if nulls.loc[c, 'dtype'].startswith('float') or nulls.loc[c, 'dtype'].startswith('int')]
cat_cols = [c for c in nulls.index if c not in num_cols]
print(f'{len(num_cols)} numeric columns:', num_cols)
print(f'{len(cat_cols)} text / categorical columns:', cat_cols)

22 numeric columns: ['Age', 'BodyweightKg', 'Squat1Kg', 'Squat2Kg', 'Squat3Kg', 'Squat4Kg', 'Best3SquatKg', 'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 'Bench4Kg', 'Best3BenchKg', 'Deadlift1Kg', 'Deadlift2Kg', 'Deadlift3Kg', 'Deadlift4Kg', 'Best3DeadliftKg', 'TotalKg', 'Dots', 'Wilks', 'Glossbrenner', 'Goodlift']
20 text / categorical columns: ['Name', 'Sex', 'Event', 'Equipment', 'AgeClass', 'BirthYearClass', 'Division', 'WeightClassKg', 'Place', 'Tested', 'Country', 'State', 'Federation', 'ParentFederation', 'Date', 'MeetCountry', 'MeetState', 'MeetTown', 'MeetName', 'Sanctioned']


Attempt-by-attempt columns are empty for about two thirds of rows: many federations only report each lifter's best lifts. `State` (79%), the 4th attempts (99%+) and `ParentFederation` (39%) are mostly empty.

## 4. Load the project columns

From here on we only load the 25 columns the project uses (`src/clean.py`, `USECOLS`).

In [5]:
df = load_raw()
print(df.shape)
print(f"{df['Name'].nunique():,} lifters | {df.groupby(['Federation', 'Date', 'MeetName'], observed=True).ngroups:,} meets | "
      f"{df['Federation'].nunique()} federations | {df['MeetCountry'].nunique()} meet countries")
print('Meet dates:', df['Date'].min().date(), 'to', df['Date'].max().date())

(4050248, 25)


1,016,942 lifters | 63,900 meets | 477 federations | 133 meet countries
Meet dates: 1964-09-05 to 2026-10-13


In [6]:
df.describe().T.round(1)

,count,mean,min,25%,50%,75%,max,std
Age,2586588.0,30.681206,0.0,20.5,27.0,38.0,105.5,13.355965
BodyweightKg,4005793.0,83.76841,15.0,66.860001,81.599998,98.300003,300.0,22.685568
Squat1Kg,1329345.0,117.986092,-555.0,95.0,147.5,197.5,560.01001,132.209564
Squat2Kg,1315488.0,103.753311,-600.01001,85.0,150.0,202.5,577.5,155.913651
Squat3Kg,1283882.0,45.171413,-600.5,-147.419998,120.0,195.0,595.0,188.017563
Best3SquatKg,2821669.0,171.073151,-508.019989,120.0,165.559998,215.0,595.0,67.030457
Bench1Kg,1927736.0,84.077232,-635.5,57.5,100.0,140.0,659.0,93.287216
Bench2Kg,1905349.0,59.780605,-635.5,37.5,95.0,140.0,530.0,118.013062
Bench3Kg,1850258.0,-11.735598,-589.98999,-130.0,-55.0,115.0,635.5,135.554352
Best3BenchKg,3594391.0,114.640442,-522.5,72.5,111.129997,147.5,659.0,52.515533


## 5. Unique values of the main categorical columns

In [7]:
for col in ['Sex', 'Event', 'Equipment', 'Tested', 'Sanctioned']:
    print(f'{col:11s}', df[col].value_counts(dropna=False).to_dict())

Sex         {'M': 2917030, 'F': 1133019, 'Mx': 199}
Event       {'SBD': 2925027, 'B': 802815, 'D': 219805, 'BD': 75966, 'S': 19960, 'SB': 4185, 'SD': 2490}
Equipment   {'Raw': 1935362, 'Single-ply': 1416353, 'Unlimited': 324522, 'Wraps': 242582, 'Multi-ply': 131310, 'Straps': 119}
Tested      {'Yes': 3087897, nan: 962351}
Sanctioned  {'Yes': 4038812, 'No': 11436}


In [8]:
pf = df['ParentFederation'].astype(str).replace('nan', '(none)').value_counts()
pf_table = pd.DataFrame({'rows': pf, 'share_%': (pf / len(df) * 100).round(1)})
pf_table.head(10)

,rows,share_%
ParentFederation,,
IPF,1531081,37.8
IPL,279902,6.9
WPC,253428,6.3
WRPF,97304,2.4
GPC,63356,1.6
WDFPF,49226,1.2
IPA,46074,1.1
GPA,31848,0.8
WUAP,21083,0.5


**IPF-affiliated meets are the largest group with a parent federation** (about 1.5 million rows). The project is scoped to these meets, because the IPF rulebook is the one the judging assistant will follow.

## 6. Duplicates

In [9]:
print('Exact duplicate rows                :', df.duplicated().sum())
same_meet = df.duplicated(['Name', 'Date', 'MeetName', 'Federation'])
print(f'Same lifter, same meet, extra rows   : {same_meet.sum():,} ({same_meet.mean():.1%})')

example = df[df.duplicated(['Name', 'Date', 'MeetName', 'Federation'], keep=False)].sort_values(['Name', 'Date']).head(4)
example[['Name', 'Date', 'MeetName', 'Event', 'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 'Best3BenchKg']]

Exact duplicate rows                : 187670


Same lifter, same meet, extra rows   : 383,462 (9.5%)


,Name,Date,MeetName,Event,Bench1Kg,Bench2Kg,Bench3Kg,Best3BenchKg
1434654,A Arun Kumar,2018-11-14,Indian Bench Press Championships,B,150.0,-160.0,-160.0,150.0
1435147,A Arun Kumar,2018-11-14,Indian Bench Press Championships,B,135.0,-137.5,-137.5,135.0
2755047,A Cudd,2025-01-18,British Equipped Championships,B,190.0,-200.0,-200.0,190.0
2755066,A Cudd,2025-01-18,British Equipped Championships,B,190.0,-200.0,-200.0,190.0


A lifter who enters two divisions at the same meet (for example *Open* and *Juniors*) appears once per division with the **same attempts**. Keeping both would count those attempts twice, so only the first row per lifter per meet is kept.

## 7. Hidden codes and special values

In [10]:
att = df[ATTEMPT_COLS]
print(f'Attempt cells recorded  : {att.notna().sum().sum():,}')
print(f'Negative (= missed)     : {(att < 0).sum().sum():,} ({(att < 0).sum().sum() / att.notna().sum().sum():.1%} of recorded attempts)')
print(f'Exactly 0 kg            : {(att == 0).sum().sum():,}  (no lift taken; treated as not attempted)')

has_attempts = att.notna().any(axis=1)
print(f'\nRows with attempt-by-attempt data: {has_attempts.sum():,} ({has_attempts.mean():.1%})')
print('Share with attempt data, IPF vs other parent federations:')
print(has_attempts.groupby(df['ParentFederation'] == 'IPF').mean().rename({True: 'IPF', False: 'other / none'}).round(3))

Attempt cells recorded  : 13,972,884
Negative (= missed)     : 3,449,926 (24.7% of recorded attempts)
Exactly 0 kg            : 0  (no lift taken; treated as not attempted)



Rows with attempt-by-attempt data: 2,117,613 (52.3%)
Share with attempt data, IPF vs other parent federations:
ParentFederation
other / none    0.457
IPF             0.631
dtype: float64


In [11]:
approx_age = (df['Age'] % 1 == 0.5)
print(f"Approximate ages (x.5)        : {approx_age.sum():,} ({approx_age.mean():.1%} of rows)")
print(f"Age missing                   : {df['Age'].isna().mean():.1%}")
print(f"Age under 10 / over 85        : {(df['Age'] < 10).sum():,} / {(df['Age'] > 85).sum():,}")
print(f"Bodyweight under 30 kg        : {(df['BodyweightKg'] < 30).sum():,}")
print(f"Negative best lifts           : {(df[BEST_COLS] < 0).any(axis=1).sum():,}")
print(f"Unsanctioned meets (rows)     : {(df['Sanctioned'] == 'No').sum():,}")

Approximate ages (x.5)        : 1,132,969 (28.0% of rows)
Age missing                   : 36.1%
Age under 10 / over 85        : 3,691 / 633
Bodyweight under 30 kg        : 1,299
Negative best lifts           : 7,837
Unsanctioned meets (rows)     : 11,436


In [12]:
extra = pd.read_csv(RAW, usecols=['Place', 'Squat4Kg', 'Bench4Kg', 'Deadlift4Kg'], low_memory=False)
place = np.where(extra['Place'].astype(str).str.isdigit(), 'placed', extra['Place'].astype(str))
print(pd.Series(place).value_counts())
print(f"\nRows with a 4th attempt: {extra[['Squat4Kg', 'Bench4Kg', 'Deadlift4Kg']].notna().any(axis=1).sum():,}")
del extra

placed    3765447
DQ         246820
NS          22254
G           13863
DD           1864
Name: count, dtype: int64

Rows with a 4th attempt: 56,407


## 8. Findings

- **4,050,248 rows × 42 columns** (22 numeric, 20 text / categorical): **1,016,942 lifters**, 63,900 meets, 477 federations, 133 meet countries, from 1964 to October 2026.
- **Attempt-by-attempt data exists for 52.3% of rows**: 63.1% of IPF rows vs 45.7% for other federations. That gives **13,972,884 recorded attempts**, of which **24.7% were missed**.
- **Hidden code:** a missed attempt is stored as a **negative weight** (e.g. `-37.5` = missed 37.5 kg). This is how the target is built.
- **Duplicates:** 383,462 extra rows (9.5%) are the same lifter at the same meet entered in more than one division, with the same attempts → keep one row per lifter per meet.
- **Special values:** 28.0% of ages are approximate (`x.5`, birth year only) and are kept; 3,691 ages under 10, 633 over 85 and 1,299 bodyweights under 30 kg are typos → set to missing. 7,837 negative best lifts are never used as an earlier best.
- **4th attempts** (56,407 rows) are record tries that don't count toward the total → excluded. **Unsanctioned meets** (11,436 rows) → dropped.
- `Place` holds codes besides placings: DQ (246,820), NS no-show (22,254), G guest (13,863), DD doping (1,864).
- **Result columns** (`Best3…Kg`, `TotalKg`, `Place`, `Dots`, `Wilks`, `Glossbrenner`, `Goodlift`) are worked out from the attempts, so they are **never used as features** (leakage, see notebook 02).
- **Scope:** IPF-affiliated meets, 1,531,081 rows (37.8%), the largest group and the rulebook the judging project follows.